# KAN-AD + DeepSVDD on SWaT with EasyTSAD
This notebook builds a **clear final hybrid pipeline** for the SWaT dataset using:
- **KAN-AD** for forecasting
- **DeepSVDD** for latent-space compactness
- **z-score + sigmoid normalization** for both components
- **alpha tuning on the validation set**
- **EasyTSAD** for final evaluation on the test set

Final anomaly score:
\[
\text{score} = \alpha \cdot \sigma(z(\text{pred\_error})) + (1-\alpha) \cdot \sigma(z(\text{svdd\_distance}))
\]

where:
- \(z(x) = (x - \mu)/(\sigma + \varepsilon)\)
- \(\sigma(\cdot)\) is the sigmoid function
- \(\alpha \in [0,1]\) is tuned on the validation set

## 1. Install dependencies and clone repositories
This follows the structure of the previous notebook while keeping the setup explicit and reproducible.

In [ ]:
!pip install --upgrade pip
!pip install torch torchinfo tqdm numpy scikit-learn toml
!pip install git+https://github.com/dawnvince/EasyTSAD.git
!rm -rf /content/KAN-AD /content/datasets

!git clone https://github.com/CSTCloudOps/KAN-AD.git /content/KAN-AD
!git clone https://github.com/CSTCloudOps/datasets.git /content/datasets

# EasyTSAD expects this dataset layout inside the workspace repo
!rm -rf /content/KAN-AD/datasets
!mv /content/datasets /content/KAN-AD/datasets

%cd /content/KAN-AD

  Cloning https://github.com/dawnvince/EasyTSAD.git to /tmp/pip-req-build-xnxh16kj
  Running command git clone --filter=blob:none --quiet https://github.com/dawnvince/EasyTSAD.git /tmp/pip-req-build-xnxh16kj
  Resolved https://github.com/dawnvince/EasyTSAD.git to commit a13519014c0334a26b0e74480881db503e6a8e0b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into '/content/KAN-AD'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 18 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 61.77 KiB | 1.01 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Cloning into '/content/datasets'...
remote: Enumerating objects: 4503, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 4503 (delta 2), reused 0 (delta 

## 2. Configure imports and patch EasyTSAD if needed
The previous notebook included a small syntax fix for `EasyTSAD.DataFactory.__init__`. We keep that safeguard here.

In [ ]:
!sed -i 's/TSData,*/TSData/g' /usr/local/lib/python3.*/dist-packages/EasyTSAD/DataFactory/__init__.py || true
!grep -n "TSData" /usr/local/lib/python3.*/dist-packages/EasyTSAD/DataFactory/__init__.py | head -n 20

import os
import sys
import glob
import json
import math
import copy
import toml
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm
from torch.utils.data import Dataset, DataLoader

repo_root = "/content/KAN-AD"
sys.path.insert(0, repo_root)

candidates = glob.glob("/content/KAN-AD/**/kanad/kanad.py", recursive=True)
print("Found KAN-AD candidates:", candidates)
if not candidates:
    raise FileNotFoundError("Could not find kanad/kanad.py inside /content/KAN-AD")

kanad_py = sorted(candidates, key=len)[0]
KANAD_PKG_DIR = os.path.dirname(kanad_py)
KANAD_ROOT_DIR = os.path.dirname(KANAD_PKG_DIR)

sys.path.insert(0, KANAD_ROOT_DIR)

from EasyTSAD.Controller import TSADController
from EasyTSAD.DataFactory import TSData
from EasyTSAD.Methods import BaseMethod
from EasyTSAD.Evaluations.Protocols import EventF1PA, PointF1PA, PointKthF1PA, PointAuprcPA

from kanad import KANAD
from kanad.kanad import KANADModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("KANAD imported successfully:", KANAD)
print("KANADModel imported successfully:", KANADModel)
print("TSADController imported successfully:", TSADController)
print("KANAD_PKG_DIR =", KANAD_PKG_DIR)
print("KANAD_ROOT_DIR =", KANAD_ROOT_DIR)

1:from .TSData import TSData
2:from .MTSData import MTSData
Found KAN-AD candidates: ['/content/KAN-AD/kanad/kanad.py']
Device: cuda
KANAD imported successfully: <class 'kanad.kanad.KANAD'>
KANADModel imported successfully: <class 'kanad.kanad.KANADModel'>
TSADController imported successfully: <class 'EasyTSAD.Controller.TSADController.TSADController'>
KANAD_PKG_DIR = /content/KAN-AD/kanad
KANAD_ROOT_DIR = /content/KAN-AD


## 3. Patch `KANADModel` helper methods
The hybrid model needs access to:
- the intermediate feature map for the SVDD branch
- the forecasting head separately

In [ ]:
def forward_feature(self, x: torch.Tensor):
    """
    x: (B, window)
    returns intermediate feature map ff of shape (B, 1, window)
    """
    res = []
    res.append(x.unsqueeze(1))

    ff = torch.cat(
        [self.orders.repeat(x.size(0), 1, 1)]
        + [torch.cos(order * x.unsqueeze(1)) for order in range(1, self.order + 1)]
        + [x.unsqueeze(1)],
        dim=1,
    )

    res.append(ff)
    ff = self.init_conv(ff)
    ff = self.bn1(ff)
    ff = self.act(ff)

    ff = self.inner_conv(ff) + res.pop()
    ff = self.bn2(ff)
    ff = self.act(ff)

    ff = self.out_conv(ff) + res.pop()
    ff = self.bn3(ff)
    ff = self.act(ff)

    return ff

def forward_head(self, ff: torch.Tensor):
    """
    ff: (B, 1, window)
    returns prediction (B, 1)
    """
    return self.final_conv(ff).squeeze(1)

KANADModel.forward_feature = forward_feature
KANADModel.forward_head = forward_head
print("KANADModel patched with forward_feature and forward_head.")

KANADModel patched with forward_feature and forward_head.


## 4. Build the multivariate sliding-window dataset for SWaT
Each sample is:
- `x`: shape `(window, F)`
- `y`: shape `(F,)`

In [ ]:
class MTSWindowDataset(Dataset):
    def __init__(self, tsData, phase, window_size):
        self.window_size = window_size

        if phase == "train":
            self.data = tsData.train
        elif phase == "valid":
            self.data = tsData.valid
        elif phase == "test":
            self.data = tsData.test
        else:
            raise ValueError("phase must be train/valid/test")

        if self.data.ndim != 2:
            raise ValueError(f"Expected 2D MTS array, got shape {self.data.shape}")

        self.N, self.F = self.data.shape
        self.sample_num = max(self.N - self.window_size, 0)

    def __len__(self):
        return self.sample_num

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.window_size]
        y = self.data[idx + self.window_size]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

print("MTSWindowDataset ready.")

MTSWindowDataset ready.


## 5. Define the final hybrid model
Key changes versus the previous notebook:
1. **Normalized score uses z-score + sigmoid**
2. **Final fusion uses** `alpha * pred + (1-alpha) * svdd`
3. **alpha is tuned on the validation set**
4. The selected alpha is then used during test-time scoring for EasyTSAD

In [ ]:
class KANAD_SVDD_AlphaTuned(BaseMethod):
    """
    Hybrid anomaly detector:
      - KAN-AD forecasting branch
      - DeepSVDD latent branch

    Final anomaly score:
      score = alpha * pred_norm + (1 - alpha) * svdd_norm

    where pred_norm and svdd_norm are computed with:
      z-score -> sigmoid
    """

    def __init__(self, params: dict) -> None:
        super().__init__()
        self.__anomaly_score = None

        self.batch_size = int(params["batch_size"])
        self.window = int(params["window"])
        self.order = int(params["order"])
        self.epochs = int(params["epochs"])
        self.lr = float(params["lr"])

        self.lambda_svdd = float(params.get("lambda_svdd", 0.1))
        self.emb_dim = int(params.get("emb_dim", 64))
        self.patience = int(params.get("patience", 5))
        self.alpha_grid = params.get("alpha_grid", [round(i / 10.0, 2) for i in range(11)])

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model = KANADModel(window=self.window, order=self.order).to(self.device)
        self.embed_head = nn.Sequential(
            nn.Linear(self.window, 128),
            nn.GELU(),
            nn.Linear(128, self.emb_dim),
        ).to(self.device)

        self.optimizer = optim.Adam(
            list(self.model.parameters()) + list(self.embed_head.parameters()),
            lr=self.lr
        )
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=5, gamma=0.75)
        self.loss = nn.MSELoss()

        self.center = None
        self.pred_mu = None
        self.pred_std = None
        self.svdd_mu = None
        self.svdd_std = None
        self.best_alpha = None
        self.best_valid_f1 = None
        self.tuned_threshold = None

    @staticmethod
    def _sigmoid_np(x: np.ndarray) -> np.ndarray:
        x = np.clip(x, -50, 50)
        return 1.0 / (1.0 + np.exp(-x))

    @staticmethod
    def _safe_std(x: np.ndarray) -> float:
        return float(np.std(x) + 1e-8)

    def _compute_embedding(self, ff: torch.Tensor) -> torch.Tensor:
        feat = ff.squeeze(1)
        return self.embed_head(feat)

    def _loader_components(self, loader):
        pred_list = []
        svdd_list = []

        self.model.eval()
        self.embed_head.eval()

        with torch.no_grad():
            for x, target in loader:
                x = x.to(self.device)
                target = target.to(self.device)

                B, W, F = x.shape
                x_1d = x.permute(0, 2, 1).reshape(B * F, W)
                t_1d = target.reshape(B * F, 1)

                ff = self.model.forward_feature(x_1d)
                out = self.model.forward_head(ff)

                pred_err = (out - t_1d).abs().reshape(B, F).max(dim=1).values
                z = self._compute_embedding(ff).reshape(B, F, -1).mean(dim=1)
                svdd = ((z - self.center) ** 2).sum(dim=1)

                pred_list.append(pred_err.detach().cpu())
                svdd_list.append(svdd.detach().cpu())

        pred_arr = torch.cat(pred_list).numpy()
        svdd_arr = torch.cat(svdd_list).numpy()
        return pred_arr, svdd_arr

    def _normalize_components(self, pred_arr: np.ndarray, svdd_arr: np.ndarray):
        pred_z = (pred_arr - self.pred_mu) / self.pred_std
        svdd_z = (svdd_arr - self.svdd_mu) / self.svdd_std
        pred_norm = self._sigmoid_np(pred_z)
        svdd_norm = self._sigmoid_np(svdd_z)
        return pred_norm, svdd_norm

    @staticmethod
    def _best_f1_threshold(scores: np.ndarray, labels: np.ndarray):
        scores = np.asarray(scores, dtype=np.float64)
        labels = np.asarray(labels).astype(np.int32)

        if len(scores) == 0:
            return 0.0, 0.0

        order = np.argsort(scores)[::-1]
        sorted_scores = scores[order]
        sorted_labels = labels[order]

        total_pos = int(sorted_labels.sum())
        if total_pos == 0:
            thr = float(sorted_scores.max()) if len(sorted_scores) else 0.0
            return 0.0, thr

        tp = 0
        fp = 0
        best_f1 = -1.0
        best_thr = float(sorted_scores[0])

        n = len(sorted_scores)
        i = 0
        while i < n:
            score_val = sorted_scores[i]
            pos_in_group = 0
            neg_in_group = 0

            while i < n and sorted_scores[i] == score_val:
                if sorted_labels[i] == 1:
                    pos_in_group += 1
                else:
                    neg_in_group += 1
                i += 1

            tp += pos_in_group
            fp += neg_in_group
            fn = total_pos - tp

            precision = tp / (tp + fp + 1e-12)
            recall = tp / (tp + fn + 1e-12)
            f1 = 2 * precision * recall / (precision + recall + 1e-12)

            if f1 > best_f1:
                best_f1 = float(f1)
                best_thr = float(score_val)

        return best_f1, best_thr

    def _extract_valid_labels(self, tsTrain: TSData):
        valid_labels = getattr(tsTrain, "valid_label", None)
        if valid_labels is None:
            raise AttributeError("TSData does not expose valid_label; alpha tuning requires validation labels.")
        valid_labels = np.asarray(valid_labels).reshape(-1)
        if len(valid_labels) <= self.window:
            raise ValueError("Validation labels are shorter than the window size.")
        return valid_labels[self.window:]

    def _tune_alpha(self, tsTrain: TSData, valid_loader):
        valid_labels = self._extract_valid_labels(tsTrain)

        pred_val, svdd_val = self._loader_components(valid_loader)
        pred_norm, svdd_norm = self._normalize_components(pred_val, svdd_val)

        if len(valid_labels) != len(pred_norm):
            min_len = min(len(valid_labels), len(pred_norm))
            valid_labels = valid_labels[:min_len]
            pred_norm = pred_norm[:min_len]
            svdd_norm = svdd_norm[:min_len]

        best_alpha = None
        best_f1 = -1.0
        best_thr = None
        history = []

        print("\nAlpha tuning on validation set")
        for alpha in self.alpha_grid:
            score = alpha * pred_norm + (1.0 - alpha) * svdd_norm
            f1, thr = self._best_f1_threshold(score, valid_labels)
            history.append({"alpha": float(alpha), "valid_best_f1": float(f1), "threshold": float(thr)})
            print(f"alpha={alpha:.2f} | valid_best_f1={f1:.6f} | threshold={thr:.6f}")

            if f1 > best_f1:
                best_f1 = f1
                best_alpha = float(alpha)
                best_thr = float(thr)

        self.best_alpha = best_alpha
        self.best_valid_f1 = best_f1
        self.tuned_threshold = best_thr
        self.alpha_history = history

        print(f"\nSelected alpha={self.best_alpha:.2f} with valid_best_f1={self.best_valid_f1:.6f}")

    def train_valid_phase(self, tsTrain: TSData):
        train_loader = DataLoader(
            MTSWindowDataset(tsTrain, "train", self.window),
            batch_size=self.batch_size,
            shuffle=True
        )
        valid_loader = DataLoader(
            MTSWindowDataset(tsTrain, "valid", self.window),
            batch_size=self.batch_size,
            shuffle=False
        )

        # ---- compute DeepSVDD center from train windows using the current initialization ----
        zs = []
        self.model.eval()
        self.embed_head.eval()
        with torch.no_grad():
            for x, _ in tqdm.tqdm(train_loader, desc="Compute SVDD center"):
                x = x.to(self.device)
                B, W, F = x.shape
                x_1d = x.permute(0, 2, 1).reshape(B * F, W)
                ff = self.model.forward_feature(x_1d)
                z = self._compute_embedding(ff).reshape(B, F, -1).mean(dim=1)
                zs.append(z.cpu())

        Z = torch.cat(zs)
        self.center = Z.mean(dim=0).to(self.device)
        self.center[(self.center.abs() < 1e-6)] = 1e-6

        best_valid = float("inf")
        patience_counter = 0
        best_state = None

        for epoch in range(self.epochs):
            self.model.train()
            self.embed_head.train()
            train_loss = 0.0

            for x, target in tqdm.tqdm(train_loader, desc=f"Train {epoch+1}"):
                x = x.to(self.device)
                target = target.to(self.device)

                B, W, F = x.shape
                x_1d = x.permute(0, 2, 1).reshape(B * F, W)
                t_1d = target.reshape(B * F, 1)

                self.optimizer.zero_grad()

                ff = self.model.forward_feature(x_1d)
                out = self.model.forward_head(ff)

                pred_loss = self.loss(out, t_1d)
                z = self._compute_embedding(ff).reshape(B, F, -1).mean(dim=1)
                svdd_loss = ((z - self.center) ** 2).sum(dim=1).mean()

                loss = pred_loss + self.lambda_svdd * svdd_loss
                loss.backward()
                self.optimizer.step()
                train_loss += loss.item()

            self.model.eval()
            self.embed_head.eval()
            valid_loss = 0.0

            with torch.no_grad():
                for x, target in tqdm.tqdm(valid_loader, desc=f"Valid {epoch+1}"):
                    x = x.to(self.device)
                    target = target.to(self.device)

                    B, W, F = x.shape
                    x_1d = x.permute(0, 2, 1).reshape(B * F, W)
                    t_1d = target.reshape(B * F, 1)

                    ff = self.model.forward_feature(x_1d)
                    out = self.model.forward_head(ff)

                    pred_loss = self.loss(out, t_1d)
                    z = self._compute_embedding(ff).reshape(B, F, -1).mean(dim=1)
                    svdd_loss = ((z - self.center) ** 2).sum(dim=1).mean()

                    loss = pred_loss + self.lambda_svdd * svdd_loss
                    valid_loss += loss.item()

            valid_loss /= max(len(valid_loader), 1)
            avg_train_loss = train_loss / max(len(train_loader), 1)
            print(f"Epoch {epoch+1} | train_loss={avg_train_loss:.6f} | valid_loss={valid_loss:.6f}")
            self.scheduler.step()

            if valid_loss < best_valid:
                best_valid = valid_loss
                patience_counter = 0
                best_state = {
                    "model": copy.deepcopy(self.model.state_dict()),
                    "embed_head": copy.deepcopy(self.embed_head.state_dict()),
                    "center": self.center.detach().cpu().clone(),
                }
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print("Early stopping")
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state["model"])
            self.embed_head.load_state_dict(best_state["embed_head"])
            self.center = best_state["center"].to(self.device)

        # ---- estimate normalization statistics on the train set ----
        pred_train, svdd_train = self._loader_components(train_loader)
        self.pred_mu = float(np.mean(pred_train))
        self.pred_std = self._safe_std(pred_train)
        self.svdd_mu = float(np.mean(svdd_train))
        self.svdd_std = self._safe_std(svdd_train)

        print("\nNormalization statistics")
        print(f"pred_mu={self.pred_mu:.6f}, pred_std={self.pred_std:.6f}")
        print(f"svdd_mu={self.svdd_mu:.6f}, svdd_std={self.svdd_std:.6f}")

        # ---- tune alpha on validation set ----
        self._tune_alpha(tsTrain, valid_loader)

    def test_phase(self, tsData: TSData):
        test_loader = DataLoader(
            MTSWindowDataset(tsData, "test", self.window),
            batch_size=self.batch_size,
            shuffle=False
        )

        scores = []
        self.model.eval()
        self.embed_head.eval()

        with torch.no_grad():
            for x, target in tqdm.tqdm(test_loader, desc="Testing"):
                x = x.to(self.device)
                target = target.to(self.device)

                B, W, F = x.shape
                x_1d = x.permute(0, 2, 1).reshape(B * F, W)
                t_1d = target.reshape(B * F, 1)

                ff = self.model.forward_feature(x_1d)
                out = self.model.forward_head(ff)

                pred_err = (out - t_1d).abs().reshape(B, F).max(dim=1).values
                z = self._compute_embedding(ff).reshape(B, F, -1).mean(dim=1)
                svdd = ((z - self.center) ** 2).sum(dim=1)

                pred_norm = torch.sigmoid((pred_err - self.pred_mu) / self.pred_std)
                svdd_norm = torch.sigmoid((svdd - self.svdd_mu) / self.svdd_std)

                alpha = 0.5 if self.best_alpha is None else self.best_alpha
                score = alpha * pred_norm + (1.0 - alpha) * svdd_norm
                scores.append(score.detach().cpu())

        scores = torch.cat(scores).numpy()
        scores[np.isnan(scores)] = 1000.0
        self.__anomaly_score = scores

    def anomaly_score(self):
        return self.__anomaly_score

    def param_statistic(self, save_file):
        stats = {
            "window": self.window,
            "order": self.order,
            "epochs": self.epochs,
            "lr": self.lr,
            "lambda_svdd": self.lambda_svdd,
            "emb_dim": self.emb_dim,
            "pred_mu": self.pred_mu,
            "pred_std": self.pred_std,
            "svdd_mu": self.svdd_mu,
            "svdd_std": self.svdd_std,
            "best_alpha": self.best_alpha,
            "best_valid_f1": self.best_valid_f1,
            "tuned_threshold": self.tuned_threshold,
            "alpha_grid": list(self.alpha_grid),
        }
        with open(save_file, "w") as f:
            json.dump(stats, f, indent=2)

print("KANAD_SVDD_AlphaTuned ready.")

KANAD_SVDD_AlphaTuned ready.


## 6. Write the EasyTSAD config file
The alpha grid is the set of candidate values searched on the validation set.

In [ ]:
config_text = """[Data_Params]
preprocess = "z-score"
diff_order = 0

[Model_Params.Default]
window = 96
order = 2
batch_size = 1024
epochs = 100
lr = 0.01
lambda_svdd = 0.1
emb_dim = 64
patience = 5
alpha_grid = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
"""

cfg_path = os.path.join(KANAD_PKG_DIR, "config_svdd_alpha_tuned.toml")
with open(cfg_path, "w") as f:
    f.write(config_text)

print("Wrote:", cfg_path)
print(open(cfg_path).read())

Wrote: /content/KAN-AD/kanad/config_svdd_alpha_tuned.toml
[Data_Params]
preprocess = "z-score"
diff_order = 0

[Model_Params.Default]
window = 96
order = 2
batch_size = 1024
epochs = 100
lr = 0.01
lambda_svdd = 0.1
emb_dim = 64
patience = 5
alpha_grid = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]



## 7. Run the EasyTSAD experiment on SWaT
This trains the hybrid model, tunes `alpha` on the validation split, tests on SWaT, and then evaluates the saved scores with EasyTSAD.

In [ ]:
gctrl = TSADController()

gctrl.set_dataset(
    dataset_type="MTS",
    dirname="/content/KAN-AD/datasets",
    datasets=["SWaT"],
)

method = "KANAD_SVDD_AlphaTuned"
training_schema = "naive"

gctrl.run_exps(
    method=method,
    training_schema=training_schema,
    cfg_path="kanad/config_svdd_alpha_tuned.toml",
)

gctrl.set_evals([
    PointF1PA(),
    EventF1PA(mode="squeeze"),
    PointKthF1PA(k=5),
    PointAuprcPA(),
])

gctrl.do_evals(method=method, training_schema=training_schema)

(2026-04-05 13:42:20,125) [INFO]: 
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚══════╝   ╚═╝          ╚═╝   ╚══════╝╚═╝  ╚═╝╚═════╝ 
                                                                      
                         
INFO:logger:
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚═════

Epoch 1 | train_loss=759.123296 | valid_loss=525.268939


Valid 2: 100%|██████████| 463/463 [00:21<00:00, 21.62it/s]


Epoch 2 | train_loss=529.070260 | valid_loss=396.361123


Valid 3: 100%|██████████| 463/463 [00:21<00:00, 21.73it/s]


...

Valid 19: 100%|██████████| 463/463 [00:21<00:00, 21.72it/s]


Epoch 19 | train_loss=120.222579 | valid_loss=440.157560


Valid 20: 100%|██████████| 463/463 [00:21<00:00, 21.83it/s]


Epoch 20 | train_loss=110.605167 | valid_loss=211.462625
Early stopping

Normalization statistics
pred_mu=7.702992, pred_std=86.030052
svdd_mu=0.001331, svdd_std=0.300371

Alpha tuning on validation set
alpha=0.00 | valid_best_f1=0.000000 | threshold=1.000000
alpha=0.10 | valid_best_f1=0.000000 | threshold=1.000000
alpha=0.20 | valid_best_f1=0.000000 | threshold=1.000000
alpha=0.30 | valid_best_f1=0.000000 | threshold=1.000000
alpha=0.40 | valid_best_f1=0.000000 | threshold=1.000000
alpha=0.50 | valid_best_f1=0.000000 | threshold=1.000000
alpha=0.60 | valid_best_f1=0.000000 | threshold=1.000000
alpha=0.70 | valid_best_f1=0.000000 | threshold=1.000000
alpha=0.80 | valid_best_f1=0.000000 | threshold=1.000000
alpha=0.90 | valid_best_f1=0.000000 | threshold=1.000000
alpha=1.00 | valid_best_f1=0.000000 | threshold=1.000000

Selected alpha=0.00 with valid_best_f1=0.000000


Testing: 100%|██████████| 440/440 [00:20<00:00, 21.53it/s]
(2026-04-05 14:08:33,803) [INFO]: Register evaluations
INFO:logger:Register evaluations
(2026-04-05 14:08:33,804) [INFO]: Perform evaluations. Method[KANAD_SVDD_AlphaTuned], Schema[naive].
INFO:logger:Perform evaluations. Method[KANAD_SVDD_AlphaTuned], Schema[naive].
(2026-04-05 14:08:33,805) [INFO]:     [Load Data (All)] DataSets: SWaT 
INFO:logger:    [Load Data (All)] DataSets: SWaT 
(2026-04-05 14:08:33,866) [INFO]:     [KANAD_SVDD_AlphaTuned] Eval dataset SWaT <<<
INFO:logger:    [KANAD_SVDD_AlphaTuned] Eval dataset SWaT <<<
(2026-04-05 14:08:33,867) [INFO]:         [SWaT] Using default margins (0, 5)
INFO:logger:        [SWaT] Using default margins (0, 5)


## 8. Load and display EasyTSAD evaluation results
This cell locates the generated evaluation files and prints the aggregate metrics.

In [ ]:
import glob

base = "/content/KAN-AD/Results/Evals"
if not os.path.exists(base):
    base = "/content/KAN-AD/KAN-AD/Results/Evals"

avg_files = glob.glob(os.path.join(base, "**", "SWaT", "avg.json"), recursive=True)
all_files = glob.glob(os.path.join(base, "**", "SWaT", "all.json"), recursive=True)

print("Found avg.json:", avg_files)
print("Found all.json:", all_files)

assert len(avg_files) > 0, "avg.json not found"
assert len(all_files) > 0, "all.json not found"

avg_path = avg_files[0]
all_path = all_files[0]

with open(avg_path, "r") as f:
    avg = json.load(f)

with open(all_path, "r") as f:
    all_scores = json.load(f)

print("\n=== AVERAGE RESULTS (global metrics) ===")
for k, v in avg.items():
    print(f"{k}: {v}")

print("\n=== PER-SERIES RESULTS ===")
print("Number of series:", len(all_scores))
print("Example entry:", list(all_scores.items())[0] if len(all_scores) > 0 else "No per-series entries")

Found avg.json: ['/content/KAN-AD/Results/Evals/KANAD_SVDD_AlphaTuned/naive/SWaT/avg.json']
Found all.json: ['/content/KAN-AD/Results/Evals/KANAD_SVDD_AlphaTuned/naive/SWaT/all.json']

=== AVERAGE RESULTS (global metrics) ===
best f1 under pa: {'f1': 0.30001287554995837, 'precision': 0.18118473507132915, 'recall': 0.871724213446237, 'threshold': 0.0}
event-based f1 under pa with mode squeeze: {'f1': 0.0022757443580504327, 'precision': 0.001145475372279496, 'recall': 0.17142857142857143, 'threshold': 0.0}
best f1 under 5-delay pa: {'f1': 0.30001287554995837, 'precision': 0.18118473507132915, 'recall': 0.871724213446237, 'threshold': 0.0}
point-based auprc pa: 0.12181680349826488

=== PER-SERIES RESULTS ===
Number of series: 1
Example entry: ('AllInOne', {'best f1 under pa': {'f1': 0.30001287554995837, 'precision': 0.18118473507132915, 'recall': 0.871724213446237, 'threshold': 0.0}, 'event-based f1 under pa with mode squeeze': {'f1': 0.0022757443580504327, 'precision': 0.0011454753722794

## 9. Load runtime statistics and save a compact thesis-friendly summary
EasyTSAD stores runtime metadata separately. This cell tries to locate the parameter statistics file created by the method, including the selected alpha.

In [ ]:
runtime_base = "/content/KAN-AD/Results/RunTime"
if not os.path.exists(runtime_base):
    runtime_base = "/content/KAN-AD/KAN-AD/Results/RunTime"

runtime_files = glob.glob(os.path.join(runtime_base, "**", "SWaT", "*.json"), recursive=True)
print("Runtime/stat files:", runtime_files[:10])

selected_runtime = None
for fp in runtime_files:
    name = os.path.basename(fp).lower()
    if "param" in name or "stat" in name or method.lower() in fp.lower():
        selected_runtime = fp
        break

runtime_info = {}
if selected_runtime and os.path.exists(selected_runtime):
    try:
        with open(selected_runtime, "r") as f:
            runtime_info = json.load(f)
    except Exception as e:
        print("Could not parse runtime info:", e)

summary_row = {
    "model": method,
    "dataset": "SWaT",
    "training_schema": training_schema,
    "config_path": "kanad/config_svdd_alpha_tuned.toml",
}

summary_row.update(runtime_info)
summary_row.update(avg)

summary_path = "/content/KANAD_SVDD_AlphaTuned_SWaT_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary_row, f, indent=2)

print("Saved summary to:", summary_path)
print("\nSummary row:")
for k, v in summary_row.items():
    print(f"{k}: {v}")

Runtime/stat files: ['/content/KAN-AD/Results/RunTime/KANAD_SVDD_AlphaTuned/naive/SWaT/time.json']
Saved summary to: /content/KANAD_SVDD_AlphaTuned_SWaT_summary.json

Summary row:
model: KANAD_SVDD_AlphaTuned
dataset: SWaT
training_schema: naive
config_path: kanad/config_svdd_alpha_tuned.toml
train_and_valid: 1544.833900624
test: 20.443746355000258
best f1 under pa: {'f1': 0.30001287554995837, 'precision': 0.18118473507132915, 'recall': 0.871724213446237, 'threshold': 0.0}
event-based f1 under pa with mode squeeze: {'f1': 0.0022757443580504327, 'precision': 0.001145475372279496, 'recall': 0.17142857142857143, 'threshold': 0.0}
best f1 under 5-delay pa: {'f1': 0.30001287554995837, 'precision': 0.18118473507132915, 'recall': 0.871724213446237, 'threshold': 0.0}
point-based auprc pa: 0.12181680349826488
